# 🚀 Amazon EMR for Data Engineering — Masterclass Notebook

Welcome to **Project 16: Amazon EMR for Data Engineering** from the *Data Engineering Diaries* by Sathiya Murthy.

### What You Will Learn in This Notebook:
1. **EMR Architecture Fundamentals**: Master, Core, and Task nodes; YARN resource negotiation; EMRFS vs HDFS.
2. **Cluster Lifecycle Management via boto3**: Programmatically provision, monitor, and terminate EMR clusters.
3. **PySpark ETL Execution on EMR**: Submitting Spark jobs with EMRFS Direct Committer and Adaptive Query Execution.
4. **EMR Steps & Automation**: Submitting Spark-submit, Hive, and Shell script steps.
5. **EMR Serverless Operations**: Running serverless PySpark jobs without managing EC2 instances.
6. **Cost Optimization & Monitoring**: Capacity-Optimized Spot fleets, Managed Scaling, and CloudWatch metrics.

---

### 1. Amazon EMR Architecture Overview
Amazon EMR (Elastic MapReduce) is a cloud big data platform for processing vast amounts of data using open-source tools such as Spark, Hive, HBase, Flink, and Presto.

- **Primary Node (Master Node)**: Coordinates the cluster. It runs the YARN ResourceManager and the HDFS NameNode. It orchestrates step execution and job tracking.
- **Core Nodes**: Run HDFS DataNode daemons and YARN NodeManagers. They execute tasks and store HDFS data.
- **Task Nodes**: Run YARN NodeManagers only. They perform computation but do not store data in HDFS. This makes them ideal for **Spot Instances**, as their termination does not result in HDFS data loss.
- **EMRFS (EMR File System)**: An extension of HDFS that allows EMR clusters to store data directly in Amazon S3. By storing data in S3 via EMRFS, storage and compute can be scaled independently, drastically reducing costs.


### 2. Programmatic Cluster Provisioning via Boto3 EMR Client
We can provision EMR clusters dynamically using Python and boto3. This allows data pipelines (orchestrated by tools like Airflow or Step Functions) to spin up transient EMR clusters, run a series of Spark steps, and terminate them automatically upon completion to save costs.


In [ ]:
import sys
import os

# Add parent directory to path so python can import src
sys.path.append(os.path.abspath(".."))

from src.cluster_config import EMRClusterConfigBuilder

# 1. Build a Spot-Optimized EMR Cluster Configuration using our builder pattern
config_builder = (
    EMRClusterConfigBuilder("notebook-demo-pyspark-cluster", release_label="emr-6.13.0")
    .with_applications(["Hadoop", "Spark", "Hive"])
    .with_logging("s3://production-emr-logs-bucket/elasticmapreduce/")
    .with_roles(service_role="EMR_DefaultRole", job_flow_role="EMR_EC2_DefaultRole")
    .with_network(subnet_id="subnet-0123456789abcdef0")
    .with_instance_fleets(
        master_types=["m5.xlarge"],
        core_types=["r5.xlarge"],
        task_types=["r5.xlarge", "m5.xlarge"],
        target_core_ondemand=2,
        target_task_spot=4
    )
    .add_bootstrap_action(
        name="InstallPipDependencies",
        s3_script_path="s3://production-emr-code-bucket/bootstrap_actions/install_dependencies.sh"
    )
    .add_configuration(
        classification="spark-defaults",
        properties={
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.shuffle.partitions": "200"
        }
    )
)

cluster_config = config_builder.build()
print("Successfully constructed EMR Config JSON!")
print(f"Cluster Name: {cluster_config['Name']}")
print(f"Target Core On-Demand Capacity: {cluster_config['Instances']['InstanceFleets'][1]['TargetOnDemandCapacity']}")
print(f"Target Task Spot Capacity: {cluster_config['Instances']['InstanceFleets'][2]['TargetSpotCapacity']}")


### 3. Distributed PySpark ETL on Local Spark Session
Before submitting PySpark scripts to a remote, expensive EMR cluster, it is a data engineering best practice to execute them locally using a local Spark session (`local[*]`) to test transformations, schema casting, filters, and output formatting. 

Let's run the ETL logic from [spark_on_emr.py](file:///d:/engineering-projects/production-data-engineering-projects/projects/16_emr/src/spark_on_emr.py) on our sample CSV data.


In [ ]:
import shutil
import os
from src.spark_on_emr import create_spark_session, run_etl

# 1. Initialize local SparkSession
spark = create_spark_session(app_name="EMR-Fundamentals-Notebook-Local", is_local=True)

# 2. Define paths
input_csv = "../sample-data/large_dataset.csv"
output_parquet = "output_parquet_notebook_test"

# Clean previous test directories if they exist
if os.path.exists(output_parquet):
    shutil.rmtree(output_parquet)

try:
    # 3. Execute PySpark ETL Pipeline
    run_etl(spark, input_csv, output_parquet)

    # 4. Read the processed Parquet output back to verify schema and data
    df_result = spark.read.parquet(output_parquet)
    print("\n" + "="*50)
    print("ETL RESULTS PREVIEW")
    print("="*50)
    df_result.show(5, truncate=False)
    
finally:
    # Always stop Spark Session to release resources
    spark.stop()
    
    # Clean up test output directory
    if os.path.exists(output_parquet):
        shutil.rmtree(output_parquet)


### 4. Hive Metastore & EMR Serverless API Operations
In enterprise workloads, EMR utilizes the **AWS Glue Data Catalog** as an external Hive Metastore. This maintains metadata tables across transient cluster runs. 

For modern workloads, **EMR Serverless** is preferred over running persistent EC2 clusters. Below we show how to instantiate the EMR Serverless Manager using Boto3 to programmatically spin up applications and submit serverless Spark runs.


In [ ]:
from src.emr_serverless import EMRServerlessManager

# Initialize the serverless management client
serverless_mgr = EMRServerlessManager(region_name="us-east-1")

print("Successfully instantiated EMRServerlessManager!")
print("\nExposed orchestration methods:")
for attr in dir(serverless_mgr):
    if not attr.startswith("_"):
        print(f" -> {attr}()")
